In [1]:
import cv2
import mediapipe as mp

# Webcam
cap = cv2.VideoCapture(0)

# MediaPipe Pose
mpPose = mp.solutions.pose

pose = mpPose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Drawing Utility
mpDraw = mp.solutions.drawing_utils

# Power Mode Variable
power_mode = False

while True:

    success, img = cap.read()

    if not success:
        break

    # Mirror Effect
    img = cv2.flip(img, 1)

    h, w, c = img.shape

    # Convert Image
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Process Pose
    results = pose.process(imgRGB)

    # Draw Pose Landmarks
    if results.pose_landmarks:

        mpDraw.draw_landmarks(
            img,
            results.pose_landmarks,
            mpPose.POSE_CONNECTIONS
        )

        # Get landmarks
        landmarks = results.pose_landmarks.landmark

        # Left wrist
        left_wrist = landmarks[15]

        # Right wrist
        right_wrist = landmarks[16]

        # Nose
        nose = landmarks[0]

        # Convert to pixels
        lw_y = int(left_wrist.y * h)
        rw_y = int(right_wrist.y * h)
        nose_y = int(nose.y * h)

        # Check if both hands are raised
        if lw_y < nose_y and rw_y < nose_y:

            power_mode = True

        else:
            power_mode = False

    # POWER MODE ACTIVATED
    if power_mode:

        # Glow Border
        cv2.rectangle(
            img,
            (10,10),
            (w-10,h-10),
            (0,255,255),
            10
        )

        # Main Text
        cv2.putText(
            img,
            "TOPE SUPER POWER ACTIVATED! ⚡",
            (40,80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.3,
            (0,255,255),
            4
        )

    else:

        cv2.putText(
            img,
            "RAISE BOTH HANDS 🙌",
            (80,60),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255,255,0),
            3
        )

    # Footer
    cv2.putText(
        img,
        "PPSL AI Pose Game",
        (120,440),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0,255,0),
        2
    )

    # Show Camera
    cv2.imshow("AI SUPERHERO POSE", img)

    # Quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release
cap.release()
cv2.destroyAllWindows()